# ask

> Answer questions about a video course using LLM tool use over the yttoc cache

In [ ]:
#| default_exp ask

## Design

`yttoc-ask` is a read-only reasoning shell. It exposes 2 thin tools (`get_summaries`, `get_xscript_range`) to an LLM via the OpenAI Responses API. The LLM decides what to fetch. Python only formats output.

See `docs/superpowers/specs/2026-04-16-yttoc-ask-design.md` for full design rationale.

In [ ]:
#| export
import json, sys
from pathlib import Path

In [ ]:
#| export
SYSTEM_PROMPT = """You are answering questions about a video course.
Use the provided tools to look up information. Do not fabricate content.
Answer in the user's language. If data is unavailable, say so."""

TOOLS = [
    {
        "type": "function",
        "name": "get_summaries",
        "description": "Return the full summaries.json for a cached video. Contains: video metadata (id, title, url, channel, duration), all sections (path, title, start/end seconds, summary, keywords, evidence with timestamp), and a full-video summary. Returns {\"error\": \"...\"} if the video is not prepared.",
        "parameters": {
            "type": "object",
            "properties": {
                "video_id": {"type": "string", "description": "YouTube video ID"}
            },
            "required": ["video_id"],
        },
    },
    {
        "type": "function",
        "name": "get_xscript_range",
        "description": "Return raw parsed transcript segments within a time range [start, end). Each segment is {start: float_seconds, end: float_seconds, text: string}. Use start/end values from get_summaries sections. Returns {\"error\": \"...\"} if captions are not available.",
        "parameters": {
            "type": "object",
            "properties": {
                "video_id": {"type": "string", "description": "YouTube video ID"},
                "start": {"type": "number", "description": "Start time in seconds"},
                "end": {"type": "number", "description": "End time in seconds"},
            },
            "required": ["video_id", "start", "end"],
        },
    },
]

ANSWER_SCHEMA = {
    "type": "json_schema",
    "name": "course_answer",
    "strict": True,
    "schema": {
        "type": "object",
        "properties": {
            "answer": {"type": "string", "description": "Synthesized answer to the question"},
            "citations": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "video_id": {"type": "string"},
                        "seconds": {"type": "integer"},
                    },
                    "required": ["video_id", "seconds"],
                    "additionalProperties": False,
                },
            },
        },
        "required": ["answer", "citations"],
        "additionalProperties": False,
    },
}